# Análisis operativa SPOT EFactor

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from collections import deque
path='Efactor.xlsx'
df=pd.read_excel(path)
df['fecha_operacion']=pd.to_datetime(df['fecha_operacion'])
df=df[df['clasificacion'].eq('SPOT')].copy()
ops=df[df['tipo_operacion'].isin(['C','V'])].copy().sort_values(['fecha_operacion','nm_operacion']).reset_index(drop=True)
ops['rate']=ops['volumen_mxn']/ops['volumen_usd']
ops['client_side']=np.where(ops['tipo_operacion']=='V','BUY_USD','SELL_USD')
ops['client_qty_signed']=np.where(ops['client_side']=='BUY_USD',ops['volumen_usd'],-ops['volumen_usd'])
ops.head()

In [ ]:
daily=ops.groupby('fecha_operacion').agg(ops=('nm_operacion','count'),buy_ops=('client_side', lambda s:(s=='BUY_USD').sum()),sell_ops=('client_side', lambda s:(s=='SELL_USD').sum()),client_net_usd=('client_qty_signed','sum'),usd_gross=('volumen_usd','sum'))
daily['same_day_two_way']=(daily['buy_ops']>0)&(daily['sell_ops']>0)
daily['net_to_gross']=daily['client_net_usd'].abs()/daily['usd_gross']
daily.head()

In [ ]:
from collections import deque
longs, shorts = deque(), deque()
matches=[]
for _, row in ops.iterrows():
    date=row['fecha_operacion']; opid=row['nm_operacion']; qty=row['client_qty_signed']; rate=row['rate']
    if qty>0:
        remaining=qty
        while remaining>1e-9 and shorts:
            sh=shorts[0]
            close_qty=min(remaining, sh['remaining'])
            pnl=(sh['open_rate']-rate)*close_qty
            matches.append({'open_date':sh['open_date'],'close_date':date,'hold_days':(date-sh['open_date']).days,'direction':'SHORT_USD','qty_usd':close_qty,'pnl_mxn':pnl})
            sh['remaining']-=close_qty; remaining-=close_qty
            if sh['remaining']<=1e-9: shorts.popleft()
        if remaining>1e-9:
            longs.append({'open_date':date,'open_rate':rate,'remaining':remaining,'open_opid':opid})
    else:
        remaining=-qty
        while remaining>1e-9 and longs:
            lg=longs[0]
            close_qty=min(remaining, lg['remaining'])
            pnl=(rate-lg['open_rate'])*close_qty
            matches.append({'open_date':lg['open_date'],'close_date':date,'hold_days':(date-lg['open_date']).days,'direction':'LONG_USD','qty_usd':close_qty,'pnl_mxn':pnl})
            lg['remaining']-=close_qty; remaining-=close_qty
            if lg['remaining']<=1e-9: longs.popleft()
        if remaining>1e-9:
            shorts.append({'open_date':date,'open_rate':rate,'remaining':remaining,'open_opid':opid})
matches_df=pd.DataFrame(matches)
matches_df.describe()

In [ ]:
print('Dias con compra y venta el mismo dia:', round(daily['same_day_two_way'].mean(),4))
print('% volumen emparejado cerrado mismo dia:', round(matches_df.loc[matches_df['hold_days']==0,'qty_usd'].sum()/matches_df['qty_usd'].sum(),4))
print('% volumen emparejado <=1 dia:', round(matches_df.loc[matches_df['hold_days']<=1,'qty_usd'].sum()/matches_df['qty_usd'].sum(),4))
print('P&L implicito total MXN:', round(matches_df['pnl_mxn'].sum(),2))